# Reduce dimensionality / remove redundant features

we have 278 columns, most of which are ECG waveform readings (V1_00–V6_79, etc.). Many of these are highly correlated, so keeping all may:

Increase computation

Cause overfitting

Make model harder to interpret

We can use correlation analysis first to remove highly correlated columns, then optionally PCA to reduce them further.

Code for correlation-based feature removal:

In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import RandomOverSampler

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import numpy as np

In [17]:
import pandas as pd

In [18]:
df=pd.read_csv("/content/arrhythmia_final.csv")

In [19]:
df.head()

,Age,Sex,Height,Weight,QRS_Dur,P-R_Int,Q-T_Int,T_Int,P_Int,QRS,...,V6270,V6271,V6272,V6273,V6274,V6275,V6276,V6277,V6278,V6279
0,75.0,0.0,190.0,80.0,91.0,193.0,371.0,174.0,121.0,-16.0,...,-0.3,0.0,9.0,-0.9,0.0,0.0,0.9,2.9,23.3,49.4
1,56.0,1.0,165.0,64.0,81.0,174.0,401.0,149.0,39.0,25.0,...,-0.5,0.0,8.5,0.0,0.0,0.0,0.2,2.1,20.4,38.8
2,54.0,0.0,172.0,95.0,138.0,163.0,386.0,185.0,102.0,96.0,...,0.9,0.0,9.5,-2.4,0.0,0.0,0.3,3.4,12.3,49.0
3,55.0,0.0,175.0,94.0,100.0,202.0,380.0,179.0,143.0,28.0,...,0.1,0.0,12.2,-2.2,0.0,0.0,0.4,2.6,34.6,61.6
4,75.0,0.0,190.0,80.0,88.0,181.0,360.0,177.0,103.0,-16.0,...,-0.4,0.0,13.1,-3.6,0.0,0.0,-0.1,3.9,25.4,62.8


In [20]:
# List of numeric columns (excluding class label)
numeric_cols = df.select_dtypes(include=['float64','int64']).columns.tolist()
numeric_cols.remove('V6279')  # remove class label (column V6279)

In [29]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

In [30]:
categorical_cols

[]

In [28]:
df['Sex']

,Sex
0,0.0
1,1.0
2,0.0
3,0.0
4,0.0
...,...
447,1.0
448,0.0
449,0.0
450,1.0


In [31]:
# only need scaling for numeric features before applying PCA or fitting a model.

In [41]:
target_col = 'V6279'


X = df[numeric_cols]
y = df[target_col]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=None
)

In [45]:
# Make sure target is integer
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [42]:
# BUILD PREPROCESSOR
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),

    ])

In [43]:
# MODEL PIPELINE
# have ~278 numeric columns.
# Many are highly correlated waveform features.
# Using n_components=0.95 is good because:
# It will automatically reduce dimensionality based on actual correlations.
# Keeps most of the information (~95% of variance).
# Makes your RandomForest or other models faster without losing much accuracy.
clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('pca', PCA(n_components=0.95, random_state=42)),   # keep ~95% variance
    ('model', RandomForestClassifier(n_estimators=2, random_state=42))
])

In [48]:
# TRAIN & EVALUATE
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)




In [49]:
# column transformer + scaling + PCA + oversampling + multiple models.

In [ ]:
# Key Decisions First:

# Oversampling should happen after train/test split, otherwise you leak information from test data into the training set.

# Scaling and PCA should be part of a pipeline, so you don’t need to manually scale or PCA the resampled data; this ensures clean preprocessing inside cross-validation or repeated experiments.

# Target column must be integer for classifiers.

# fl

In [69]:

# 1. Train/Test split


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=None
)
# Make sure target is integer
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# 2. Oversample to handle imbalance (train only)

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)


# 3. Preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        # No categorical encoding needed since 'Sex' is already numeric
    ]
)

In [70]:
# 4. Models & pipeline

models = {
    'KNN': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(solver='saga', max_iter=100, random_state=42), #1000
    'Decision Tree': DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42),#max_depth=5
    'Linear SVC': LinearSVC(C=0.01, max_iter=500, random_state=42),#max_iter=5000
    'Kernel SVC': SVC(kernel='sigmoid', C=10, gamma=0.001, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, criterion='gini', max_features=100, max_depth=10, max_leaf_nodes=30, random_state=42
    )#n_estimators=300
}

In [71]:
# Initialize results dataframe
result = pd.DataFrame(columns=['Model','Train Accuracy','Test Accuracy'])

In [72]:
# ----------------------------
# 5. Train, PCA + evaluate
# ----------------------------

# Initialize an empty list to store results
results_list = []

for name, model in models.items():
    # Pipeline: preprocess -> PCA -> model
    clf = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('pca', PCA(n_components=0.95, random_state=42)),  # retain 95% variance
        ('model', model)
    ])

    # Fit on oversampled training set
    clf.fit(X_train_res, y_train_res)

    # Evaluate
    train_acc = accuracy_score(y_train_res, clf.predict(X_train_res))
    test_acc = accuracy_score(y_test, clf.predict(X_test))

    # Store results in the list
    results_list.append({'Model': name, 'Train Accuracy': train_acc, 'Test Accuracy': test_acc})

# Convert the list of results to a DataFrame after the loop
result = pd.DataFrame(results_list)

# Show results
print(result)

                 Model  Train Accuracy  Test Accuracy
0                  KNN        0.814186       0.022059
1  Logistic Regression        0.919081       0.044118
2        Decision Tree        0.369630       0.014706
3           Linear SVC        0.734266       0.022059
4           Kernel SVC        0.863137       0.044118
5        Random Forest        0.656344       0.036765


In [58]:

# Pipeline handles scaling + PCA + model, no manual preprocessing needed.
# Results of all models are stored in a single dataframe for comparison.

result

,Model,Train Accuracy,Test Accuracy
0,KNN,0.814186,0.022059
1,Logistic Regression,0.491508,0.022059
2,Decision Tree,0.025974,0.000000
3,Linear SVC,0.650350,0.014706
4,Kernel SVC,0.863137,0.044118
5,Random Forest,0.175824,0.007353


In [ ]:
# oversampling + PCA + pipelines + cross-validation + weighted metrics and produce a clean results DataFrame

In [ ]:
# Oversampling for imbalanced classes

# ColumnTransformer for preprocessing

# PCA for dimensionality reduction (~95% variance)

# Pipelines for each model

# Stratified 5-fold cross-validation

# Weighted metrics: Accuracy, F1, Precision, Recall

# Clean results DataFrame

# Here’s a full, ready-to-run ve

In [62]:

# 6. Cross-validation setup

from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score
scoring = {
    'accuracy': 'accuracy',
    'f1_weighted': make_scorer(f1_score, average='weighted'),
    'precision_weighted': make_scorer(precision_score, average='weighted'),
    'recall_weighted': make_scorer(recall_score, average='weighted')
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [66]:
import warnings
from sklearn.exceptions import ConvergenceWarning
# Ignore all convergence warnings
warnings.filterwarnings("ignore", category=UserWarning)           # generic user warnings
warnings.filterwarnings("ignore", category=FutureWarning)         # future warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)    # sklearn convergence

In [73]:

# 6. Train + Evaluate all models

results_list = []
models = {

    'Logistic Regression': LogisticRegression(solver='saga', max_iter=100, random_state=42), #1000


}

for name, model in models.items():
    # Build pipeline
    clf = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('pca', PCA(n_components=0.95, random_state=42)),
        ('model', model)
    ])

    # Cross-validate
    cv_results = cross_validate(clf, X_train_res, y_train_res, cv=cv, scoring=scoring)

    results_list.append({
        'Model': name,
        'CV_Accuracy': cv_results['test_accuracy'].mean(),
        'CV_F1': cv_results['test_f1_weighted'].mean(),
        'CV_Precision': cv_results['test_precision_weighted'].mean(),
        'CV_Recall': cv_results['test_recall_weighted'].mean()
    })



In [74]:

# 7. Create results DataFrame

result_df = pd.DataFrame(results_list)
result_df = result_df.sort_values(by='CV_F1', ascending=False).reset_index(drop=True)


print(result_df)

                 Model  CV_Accuracy     CV_F1  CV_Precision  CV_Recall
0  Logistic Regression     0.789194  0.772468      0.803476   0.789194


#

# Key observations:

1- Severe overfitting

Train accuracies are moderate to high (0.37–0.92), but test accuracies are extremely low (0–0.04).

This is a classic sign that your model has memorized the training data but fails to generalize.